# Clase 4 · 03 · Tiempo, watermarks y ventanas

Cada evento tiene dos tiempos:

- **Event time** (`event_ts`): cuándo ocurrió la compra.
- **Processing time**: cuándo llega a nuestro sistema.

Para contar "compras cada 5 minutos" agrupamos por **event time** en **ventanas**. Pero los eventos pueden llegar tarde y desordenados: ¿cuándo damos por cerrada una ventana? El **watermark** responde: *"ya no espero datos más viejos que (máximo event time visto − 10 minutos)"*.

Usamos los eventos válidos que dejó el notebook 02 y los volvemos a "hacer llegar" en las mismas cinco llegadas. Cuatro consultas streaming procesan cada llegada:

| Consulta | Ventana | Modo de salida |
|---|---|---|
| `tumbling` | Fija de 5 min, sin solaparse | **append**: emite cada ventana una sola vez, cuando cierra |
| `hopping` | 10 min de ancho, avanza cada 5 min (se solapan) | append |
| `session` | Por cliente; se cierra tras 5 min sin compras | append |
| `tumbling_update` | Fija de 5 min | **update**: emite la ventana cada vez que cambia |

> El duplicado `e002` se descarta acá antes de llegar (lo trabajamos en el notebook 04). `late_bad` **no** se descarta: vamos a ver qué hace el watermark con él.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_rules

In [ ]:
%run ../common/streaming_lab

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
spark.conf.set("spark.sql.session.timeZone", "UTC")
customers = qualified_table(config, "silver_customers")
products = qualified_table(config, "silver_products")
for table in [customers, products]:
    assert spark.catalog.tableExists(table), f"Falta {table}. Ejecutá la clase 2."
T = lambda name: qualified_table(config, name)

In [ ]:
silver_table = T("stream_silver_classified")
assert spark.catalog.tableExists(silver_table), "Ejecutá primero 02_ingesta_y_enriquecimiento"

LAB = "ventanas"
tables = ["stream_arrivals", "stream_gold_tumbling", "stream_gold_hopping", "stream_gold_sessions", "stream_gold_updates"]
reset_lab(spark, dbutils, config, LAB, tables)
arrivals_table = T("stream_arrivals")
(spark.table(silver_table).where("quality_reason IS NULL").limit(0)
 .select("event_id", "event_ts", "customer_id", "amount", "payment_channel", "event_type", "source_batch_id")
 .write.format("delta").saveAsTable(arrivals_table))
WATERMARK = "10 minutes"

## 1. Las cuatro consultas

In [ ]:
purchase = F.col("event_type") == "purchase"

def metrics():
    return [F.sum(F.when(purchase, 1).otherwise(0)).alias("compras"),
            F.sum(F.when(purchase, F.col("amount")).otherwise(0)).cast("decimal(18,2)").alias("monto")]

def events():
    return spark.readStream.table(arrivals_table).withWatermark("event_ts", WATERMARK)

def tumbling():
    return (events().groupBy(F.window("event_ts", "5 minutes"), "payment_channel").agg(*metrics())
            .where("compras > 0")
            .select(F.col("window.start").alias("inicio"), F.col("window.end").alias("fin"), "payment_channel", "compras", "monto"))

def hopping():
    return (events().groupBy(F.window("event_ts", "10 minutes", "5 minutes")).agg(*metrics())
            .where("compras > 0")
            .select(F.col("window.start").alias("inicio"), F.col("window.end").alias("fin"), "compras", "monto"))

def sessions():
    return (events().groupBy(F.session_window("event_ts", "5 minutes"), "customer_id").agg(*metrics())
            .where("compras > 0")         # el control no es una compra: su sesión se descarta al final
            .select(F.col("session_window.start").alias("inicio"), F.col("session_window.end").alias("fin"),
                    "customer_id", "compras", "monto"))

def updates_writer(arrival):
    target = T("stream_gold_updates")
    def write(batch_df, batch_id):
        (batch_df.where("compras > 0")
         .select(F.col("window.start").alias("inicio"), F.col("window.end").alias("fin"), "payment_channel", "compras", "monto",
                 F.lit(arrival).alias("emitida_en_llegada"))
         .write.format("delta").mode("append").saveAsTable(target))
    return write

def tumbling_raw():
    return events().groupBy(F.window("event_ts", "5 minutes"), "payment_channel").agg(*metrics())

checkpoint = lambda name: lab_path(config, LAB, "checkpoints", name)

## 2. Cinco llegadas

En cada llegada agregamos los eventos de ese lote a la tabla de entrada y ejecutamos las cuatro consultas. Registramos el máximo event time visto, el watermark resultante, cuántas filas descartó el watermark por tardías y qué ventanas ya fueron emitidas.

In [ ]:
log, previous_watermark = [], None
for arrival, batch_id in enumerate(["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"], start=1):
    new_events = (spark.table(silver_table)
                  .where(f"quality_reason IS NULL AND source_batch_id = '{batch_id}'")
                  .select("event_id", "event_ts", "customer_id", "amount", "payment_channel", "event_type", "source_batch_id")
                  .dropDuplicates(["event_id"])
                  .join(spark.table(arrivals_table).select("event_id"), "event_id", "left_anti"))
    new_events.write.format("delta").mode("append").saveAsTable(arrivals_table)

    p_tumbling = run_stream(tumbling(), checkpoint("tumbling"), table=T("stream_gold_tumbling"))
    run_stream(hopping(), checkpoint("hopping"), table=T("stream_gold_hopping"))
    run_stream(sessions(), checkpoint("sessions"), table=T("stream_gold_sessions"))
    run_stream(tumbling_raw(), checkpoint("tumbling_update"), mode="update", foreach_batch=updates_writer(arrival))

    max_event = spark.table(arrivals_table).agg(F.date_format(F.max("event_ts"), "HH:mm")).first()[0]
    late = (spark.table(arrivals_table).where(f"source_batch_id = '{batch_id}'")
            .where(F.date_format("event_ts", "HH:mm") < F.lit(previous_watermark)).count()) if previous_watermark else 0
    previous_watermark = (spark.table(arrivals_table)          # watermark = máximo event time − 10 minutos
                          .agg(F.date_format(F.max("event_ts") - F.expr(f"INTERVAL {WATERMARK}"), "HH:mm")).first()[0])
    log.append({"llegada": arrival, "lote": batch_id,
                "máximo event time visto": max_event,
                "watermark al terminar": previous_watermark,
                "eventos más viejos que el watermark": late,
                "ventanas tumbling emitidas (total)": spark.table(T("stream_gold_tumbling")).count()})
log = pd.DataFrame(log)
display(log)

**Para observar:**
- Después de las llegadas 1 y 2 **no hay ventanas emitidas** en modo append: el watermark todavía no superó el fin de ninguna ventana.
- En la llegada 3 aparecen compras de las 12:25–12:26. El watermark salta a 12:16 y **se cierran** las ventanas de 12:00–12:10.
- En la llegada 4 llega `late_bad` (12:02), más viejo que el watermark (12:16): se **descarta** y no suma en ninguna ventana.
- El control de la llegada 5 (12:45) lleva el watermark a 12:35 y cierra la ventana de 12:25–12:30.

## 3. Event time vs. llegada

Cada punto es un evento del bloque base, ubicado según la llegada en que se procesó (eje x) y la hora en que ocurrió (eje y). La línea escalonada es el watermark al terminar cada llegada.

In [ ]:
minute_of = lambda c: F.hour(c) * 60 + F.minute(c) - 12 * 60      # minutos desde las 12:00 (UTC)
base = (spark.table(arrivals_table).where(F.col("event_id").endswith("_000"))
        .select("event_id", "source_batch_id", minute_of("event_ts").alias("minuto")).toPandas())
base["llegada"] = base["source_batch_id"].str[-1].astype(int)
wm = [int(w[:2]) * 60 + int(w[3:]) - 12 * 60 if w else None for w in log["watermark al terminar"]]

fig, ax = plt.subplots(figsize=(10, 5))
ax.step(log["llegada"], [m if m is not None else -10 for m in wm], where="post", color="#7c3aed", lw=2, label="watermark")
for row in base.itertuples():
    name = row.event_id.rsplit("_", 1)[0]
    color = "#dc2626" if name == "late_bad" else "#f59e0b" if name == "late_ok" else "#94a3b8" if name == "clock" else "#2563eb"
    marker = "X" if name == "late_bad" else "o"
    ax.scatter(row.llegada, row.minuto, s=110, color=color, marker=marker, zorder=3)
    ax.annotate(name, (row.llegada, row.minuto), xytext=(8, -3), textcoords="offset points", fontsize=8)
ax.set_xticks(range(1, 6))
ax.set_xlabel("Llegada (processing time)")
ax.set_yticks(range(-10, 50, 5), [f"{(12 * 60 + m) // 60}:{(12 * 60 + m) % 60:02d}" for m in range(-10, 50, 5)])
ax.set_ylabel("Event time")
ax.set_title("Los eventos por debajo del watermark llegan demasiado tarde")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.show()

## 4. Tumbling, hopping y session sobre los mismos eventos

Las tres formas de cortar el tiempo de la teoría. Tumbling y hopping suman las compras de todos los bloques; en session mostramos los dos clientes del bloque base.

> El control (12:45) **no se filtra antes** de agrupar: si se descartara antes, el watermark no avanzaría y las últimas ventanas nunca se cerrarían.

In [ ]:
as_minutes = lambda df: df.withColumn("inicio", minute_of("inicio")).withColumn("fin", minute_of("fin"))
tumbling_pd = as_minutes(spark.table(T("stream_gold_tumbling")).groupBy("inicio", "fin").agg(F.sum("compras").alias("compras"))
                         ).orderBy("inicio").toPandas()
hopping_pd = as_minutes(spark.table(T("stream_gold_hopping"))).orderBy("inicio").toPandas()
first_block_customers = [r.customer_id for r in spark.table(arrivals_table).where(F.col("event_id").endswith("_000"))
                         .where("event_type = 'purchase'").select("customer_id").distinct().collect()]
sessions_pd = (as_minutes(spark.table(T("stream_gold_sessions")).where(F.col("customer_id").isin(first_block_customers)))
               .orderBy("customer_id", "inicio").toPandas())

def minutes(value):
    return value

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for ax, frame, title, color in [(axes[0], tumbling_pd, "Tumbling 5 min (todas las compras)", "#2563eb"),
                                (axes[1], hopping_pd, "Hopping 10 min, salto 5 min (todas las compras)", "#16a34a")]:
    for i, row in enumerate(frame.itertuples()):
        y = i % 2 if title.startswith("Hopping") else 0
        ax.barh(y, minutes(row.fin) - minutes(row.inicio), left=minutes(row.inicio), height=0.6, color=color, alpha=0.6, edgecolor="black")
        ax.text((minutes(row.inicio) + minutes(row.fin)) / 2, y, int(row.compras), ha="center", va="center", fontsize=9)
    ax.set_yticks([])
    ax.set_title(title, loc="left", fontsize=10)
for i, (customer, group) in enumerate(sessions_pd.groupby("customer_id")):
    for row in group.itertuples():
        axes[2].barh(i, minutes(row.fin) - minutes(row.inicio), left=minutes(row.inicio), height=0.6, color="#f59e0b", edgecolor="black")
        axes[2].text((minutes(row.inicio) + minutes(row.fin)) / 2, i, f"{int(row.compras)} compras", ha="center", va="center", fontsize=8)
axes[2].set_yticks(range(sessions_pd["customer_id"].nunique()), [f"cliente {c}" for c in sorted(sessions_pd["customer_id"].unique())])
axes[2].set_title("Session: se cierra tras 5 min sin compras del cliente", loc="left", fontsize=10)
ticks = range(0, 40, 5)
axes[2].set_xticks(ticks, [f"12:{m:02d}" for m in ticks])
axes[2].set_xlabel("Event time")
fig.tight_layout()
plt.show()

## 5. Descartar o corregir: append vs. update

Seguimos la ventana **[12:00, 12:05) del canal card**. `late_ok` (12:03) pertenece a ella pero llegó en la llegada 2.

- En **update** la ventana se emite apenas tiene datos (llegada 1) y se **vuelve a emitir corregida** cuando llega `late_ok`. El consumidor debe tolerar recibir la misma ventana dos veces.
- En **append** se emite **una sola vez**, con el valor final, recién cuando el watermark la cierra. Más simple, pero con más demora.

In [ ]:
window_filter = (F.hour("inicio") == 12) & (F.minute("inicio") == 0) & (F.col("payment_channel") == "card")
updates = (spark.table(T("stream_gold_updates")).where(window_filter)
           .groupBy("emitida_en_llegada").agg(F.sum("compras").alias("compras"), F.sum("monto").alias("monto"))
           .withColumn("modo", F.lit("update")).orderBy("emitida_en_llegada").toPandas())
final = spark.table(T("stream_gold_tumbling")).where(window_filter).select("compras", "monto").first()
first_append = next(row["llegada"] for _, row in log.iterrows() if row["ventanas tumbling emitidas (total)"] > 0)
appended = pd.DataFrame([{"emitida_en_llegada": first_append, "compras": final.compras, "monto": final.monto, "modo": "append"}])
display(pd.concat([updates, appended], ignore_index=True))

## Resumen

- Las ventanas se calculan por **event time**, no por el momento de llegada.
- El **watermark** = máximo event time visto − 10 minutos. Avanza sólo cuando llegan eventos nuevos.
- Elegir el watermark es un trade-off entre **latencia** (cerrar antes) y **completitud** (esperar tardíos).
- Un evento más viejo que el watermark se **descarta** (`late_bad`); uno que llega dentro del margen se **acepta** (`late_ok`).
- **Append** emite resultados finales una vez; **update** emite antes y corrige.